In [ ]:
import pandas as pd
import numpy as np
import re
import os
from datetime import datetime

In [ ]:
# ==============================================================================
# SETUP MONTHLY OUTPUT FOLDER
# ==============================================================================
# Generates a folder name based on the current month and year (e.g., "Output_May_2026")
current_month = datetime.now().strftime("%B_%Y")
output_dir = f"Output_{current_month}"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"📁 Created new directory for this month's reports: {output_dir}/")
else:
    print(f"📁 Directory already exists: {output_dir}/")

In [ ]:
# ==============================================================================
# Helper function to catch all forms of blank cells (including pandas 'nan', 'n/a')
# ==============================================================================
def is_blank(series):
    return series.isna() | series.astype(str).str.strip().str.lower().isin(['', 'nan', 'n/a', 'na'])

In [ ]:
# ==============================================================================
# LOAD ALL DATA FILES
# ==============================================================================
print("Loading datasets...")

def load_csv(file_name):
    try:
        # Try UTF-8 with BOM first (handles standard files and Excel's new UTF-8 format)
        return pd.read_csv(file_name, encoding='utf-8-sig')
    except UnicodeDecodeError:
        # If it encounters older Excel special characters, fall back to latin-1
        return pd.read_csv(file_name, encoding='latin-1')

# Load the CSVs
current_month_data = load_csv("this_month.csv")
last_month = load_csv("last_month.csv")
dbom = load_csv("dbom_tracker.csv")
search_terms = load_csv("search_terms.csv")

# Load the Compliance Tracker as an Excel file
compliance = pd.read_excel("compliance_tracker.xlsx")

# Ensure column names match requested case format
current_month_data.rename(columns={'MARKETING_LABEL': 'marketing_label', 'REPORT_NAME': 'report_name'}, inplace=True)
last_month.rename(columns={'MARKETING_LABEL': 'marketing_label', 'REPORT_NAME': 'report_name', 'Video Type': 'video type'}, inplace=True)

In [ ]:
# ==============================================================================
# STEP 1: Compare current_month_data to last_month based on VIDEO_ID
# ==============================================================================
print("Step 1: Merging with last month's data...")
lm_cols = ['VIDEO_ID', 'ISRC', 'marketing_label', 'report_name', 'video type', 'Workability', 'dBOM', 'Metric', 'GPID']
final_df = pd.merge(current_month_data, last_month[lm_cols], on='VIDEO_ID', how='left', suffixes=('', '_lm'))

for col in ['ISRC', 'marketing_label', 'report_name']:
    final_df[col] = final_df[col].combine_first(final_df[col + '_lm'])
    final_df.drop(columns=[col + '_lm'], inplace=True)

In [ ]:
# ==============================================================================
# STEP 2: Compare to dbom_tracker based on ISRC
# ==============================================================================
print("Step 2: Updating fields from dBOM tracker...")
dbom_cols = ['ISRC', 'GPID', 'Metric', 'Sub Type', 'Report Name']
dbom_subset = dbom[dbom_cols].copy()
dbom_subset.rename(columns={'Sub Type': 'video type', 'Report Name': 'report_name'}, inplace=True)
dbom_subset = dbom_subset.drop_duplicates(subset=['ISRC'], keep='first')

final_df = pd.merge(final_df, dbom_subset, on='ISRC', how='left', suffixes=('', '_dbom'))

for col in ['GPID', 'Metric', 'video type', 'report_name']:
    final_df[col] = np.where(final_df[col + '_dbom'].notnull(), final_df[col + '_dbom'], final_df[col])
    final_df.drop(columns=[col + '_dbom'], inplace=True)

isrc_match = final_df['ISRC'].isin(dbom['ISRC'])
final_df['Workability'] = np.where(isrc_match, 'Y', final_df['Workability'])
final_df['dBOM'] = np.where(isrc_match, 'Y', final_df['dBOM'])

In [ ]:
# ==============================================================================
# STEP 3: Append missing dbom_tracker data to Current Month Data
# ==============================================================================
print("Step 3: Appending missing rows from dBOM tracker...")
existing_isrcs = final_df['ISRC'].dropna().unique()
existing_gpids = final_df['GPID'].dropna().unique()

new_dbom = dbom[~dbom['ISRC'].isin(existing_isrcs) & ~dbom['GPID'].isin(existing_gpids)].copy()
new_dbom = new_dbom.drop_duplicates(subset=['ISRC'])

new_dbom.rename(columns={
    'ARTIST': 'CHANNEL_DISPLAY_NAME',
    'dbom': 'NEW_DATE',
    'MARKETING_LABEL': 'marketing_label',
    'Report Name': 'report_name',
    'Sub Type': 'video type'
}, inplace=True)

new_dbom['VIDEO_ID'] = 'n/a'
new_dbom['CHANNEL_ID'] = 'n/a'
new_dbom['VIEW_COUNT'] = 'n/a'
new_dbom['TIME'] = 'n/a'
new_dbom['Workability'] = 'Y'
new_dbom['dBOM'] = 'Y'
new_dbom['HISTORICAL_RANK'] = 'n/a'

expected_cols = final_df.columns.tolist()
new_rows = new_dbom[[col for col in expected_cols if col in new_dbom.columns]]
final_df = pd.concat([final_df, new_rows], ignore_index=True)

In [ ]:
# ==============================================================================
# STEP 4: Compare with compliance_tracker
# ==============================================================================
print("Step 4: Applying rules from Compliance tracker...")
col_a = compliance.columns[0]
col_b = compliance.columns[1]
col_c = compliance.columns[2]
col_d = compliance.columns[3]

cond_not_y = (final_df['dBOM'] != 'Y')

match_a = final_df['VIDEO_ID'].isin(compliance[col_a].dropna())
match_b = final_df['CHANNEL_ID'].isin(compliance[col_b].dropna())
match_c = final_df['CHANNEL_DISPLAY_NAME'].isin(compliance[col_c].dropna())

drop_mask = (match_a | match_b | match_c) & cond_not_y
final_df = final_df[~drop_mask].copy()

match_d = final_df['VIDEO_ID'].isin(compliance[col_d].dropna())
final_df.loc[match_d, 'Workability'] = 'Y'
final_df.loc[match_d, 'dBOM'] = 'Y'

metric_blank = is_blank(final_df['Metric'])
final_df.loc[match_d & metric_blank, 'Metric'] = "Delivered after street date"

In [ ]:
# ==============================================================================
# STEP 5: Compare to search_terms
# ==============================================================================
print("Step 5: Applying Title searches for Video Type & dropping rules...")
col_a_kw = search_terms.iloc[:, 0].dropna().astype(str).tolist()
col_f_kw = search_terms.iloc[:, 5].dropna().astype(str).tolist()
col_g_kw = search_terms.iloc[:, 6].dropna().astype(str).tolist()
col_h_kw = search_terms.iloc[:, 7].dropna().astype(str).tolist()

col_a_kw = [kw.strip('*') for kw in col_a_kw if kw.strip('*') != '']

def make_pattern(kw_list):
    sorted_kw = sorted(kw_list, key=len, reverse=True)
    return '|'.join([re.escape(kw) for kw in sorted_kw if kw.strip() != ''])

pat_f = make_pattern(col_f_kw)
pat_g = make_pattern(col_g_kw)
pat_h = make_pattern(col_h_kw)
pat_a = make_pattern(col_a_kw)

if pat_f:
    blank_vt = is_blank(final_df['video type'])
    match_f = final_df['TITLE'].str.contains(pat_f, case=False, na=False)
    final_df.loc[match_f & blank_vt, 'video type'] = 'Lyric Video'

if pat_g:
    blank_vt = is_blank(final_df['video type'])
    match_g = final_df['TITLE'].str.contains(pat_g, case=False, na=False)
    final_df.loc[match_g & blank_vt, 'video type'] = 'Live Performance'

if pat_h:
    blank_vt = is_blank(final_df['video type'])
    match_h = final_df['TITLE'].str.contains(pat_h, case=False, na=False)
    final_df.loc[match_h & blank_vt, 'video type'] = 'Music Video'

if pat_a:
    match_a = final_df['TITLE'].str.contains(pat_a, case=False, na=False)
    cond_not_y = final_df['dBOM'] != 'Y'
    drop_mask = match_a & cond_not_y
    final_df = final_df[~drop_mask].copy()

In [ ]:
# ==============================================================================
# STEP 6: Default missing values for Workability, dBOM, Metric & Video Type
# ==============================================================================
print("Step 6: Defaulting missing values...")

blank_vid = is_blank(final_df['video type'])
is_dbom_y = final_df['dBOM'] == 'Y'
final_df.loc[blank_vid & is_dbom_y, 'video type'] = 'Music Video'

final_df.loc[is_blank(final_df['Workability']), 'Workability'] = 'N'
final_df.loc[is_blank(final_df['dBOM']), 'dBOM'] = 'N'
final_df.loc[is_blank(final_df['Metric']), 'Metric'] = 'Not dBOM'

In [ ]:
# ==============================================================================
# FINAL EXPORT FIXES: Preserve 'n/a' strings & fix Special Character Encoding
# ==============================================================================
print("Finalizing Master File...")

cols_to_fill = ['VIDEO_ID', 'CHANNEL_ID', 'VIEW_COUNT', 'TIME', 'HISTORICAL_RANK']
for col in cols_to_fill:
    if col in final_df.columns:
        final_df[col] = final_df[col].fillna('n/a')

# Dynamically generate the filename with the full month and year
month_name_full = datetime.now().strftime("%B")
report_year = datetime.now().strftime("%Y")
master_filename = f'This_Month_Data_Processed_Final_{month_name_full}_{report_year}.csv'

# Map the final output to the new monthly directory
output_file = os.path.join(output_dir, master_filename)
final_df.to_csv(output_file, index=False, encoding='utf-8-sig')

In [ ]:
# # ==============================================================================
# # STEP 7: Create Two Filtered Exports
# # ==============================================================================
# print("Step 7: Creating Filtered Exports...")

# allowed_reports = [
#     "10K", "300 Ent", "ADA - International Only", "ADA - US Only", "ADA Argentina", 
#     "ADA Australia", "ADA Benelux", "ADA Brazil", "ADA Canada", "ADA Chile", "ADA China", 
#     "ADA China (Beijing)", "ADA Colombia", "ADA France", "ADA Hong Kong", "ADA India", 
#     "ADA Indonesia", "ADA Int", "ADA Italy", "ADA Japan", "ADA Korea", "ADA Latina", 
#     "ADA Mexico", "ADA New Zealand", "ADA Norway", "ADA Peru", "ADA Russia", "ADA Singapore", 
#     "ADA South Asia", "ADA Spain", "ADA Taiwan", "ADA Thailand", "ADA UK", "ADA US", 
#     "ADA Vietnam", "Argentina", "ATL UK", "ATL US", "Australia", "Benelux", "Brazil", 
#     "Canada", "Chile", "China", "Classics", "Czech Rep", "Denmark", "Finland", "France", 
#     "Germany", "Hong Kong", "India", "Indonesia", "Ireland", "Italy", "Japan", "Korea", 
#     "Latina", "Malaysia", "Mexico", "Middle East", "Nashville", "New Zealand", "Nonesuch", 
#     "Norway", "Philippines", "PLG", "Poland", "Portugal", "Regional Asia", "Rhino UK", 
#     "Rhino US", "Russia", "Singapore", "South Africa", "Spain", "Spinnin", "Sweden", 
#     "Taiwan", "Thailand", "Turkey", "Vietnam", "WR UK", "WR US"
# ]

# allowed_videos = ['Music Video', 'Lyric Video', 'Live Video', 'Live Performance']

# clean_vid = final_df['video type'].astype(str).str.strip().str.lower()
# allowed_vids_clean = [v.lower() for v in allowed_videos]
# mask_vid = clean_vid.isin(allowed_vids_clean)

# clean_rep = final_df['report_name'].astype(str).str.strip().str.lower()
# allowed_reps_clean = [r.lower() for r in allowed_reports]
# mask_rep = clean_rep.isin(allowed_reps_clean)

# # Date Masks
# parsed_dates = pd.to_datetime(final_df['NEW_DATE'], errors='coerce')
# is_2026_dt = parsed_dates.dt.year == 2026
# is_2026_str = final_df['NEW_DATE'].astype(str).str.contains('2026', na=False)

# is_2026 = is_2026_dt | is_2026_str
# is_older = ~is_2026

# rank_num = pd.to_numeric(final_df['HISTORICAL_RANK'], errors='coerce')
# valid_rank = (rank_num >= 1) & (rank_num <= 25000)

In [ ]:
# # =========================================
# # APPLY EXACT USER CRITERIA & SAVE SEPARATELY
# # =========================================

# # Map the sub-reports to the new monthly directory
# filtered_1_file = os.path.join(output_dir, 'This_Month_Data_Filtered_2026_Only.csv')
# report_1_df = final_df[mask_vid & mask_rep & is_2026].copy()
# report_1_df.to_csv(filtered_1_file, index=False, encoding='utf-8-sig')

# filtered_2_file = os.path.join(output_dir, 'This_Month_Data_Filtered_Historical.csv')
# report_2_df = final_df[mask_vid & mask_rep & is_older & valid_rank].copy()
# report_2_df.to_csv(filtered_2_file, index=False, encoding='utf-8-sig')

In [ ]:
print(f"Success! Master file saved as '{output_file}'")
# print(f"Report 1 (Current Year) saved as '{filtered_1_file}' ({len(report_1_df)} rows)")
# print(f"Report 2 (Historical) saved as '{filtered_2_file}' ({len(report_2_df)} rows)")